# Computing Project 
## 3-Particle Quantum Mechanics

##### connecting to kernel

In [78]:
print('connected')

connected


**Importing Necessary Packages**


In [79]:
import warnings
warnings.filterwarnings('ignore')

In [80]:
import numpy as np
import scipy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.special import factorial
import pandas



#### Defining things

In [81]:
electron_charge = 14.4 ##eVA
r0 = 0.52917 ##A

**Generating n states $\psi$**

In [82]:
hyll_states = [[0,0,0], [0,0,1], [0,2,0]]
one_state_basis = [[0,0,0]]
test_kappa = 1.2

In [83]:
chosen_states = one_state_basis
chosen_num = len(chosen_states)
chosen_kappa = test_kappa

#### Making Matrices N, T, W and C


In [84]:
def generate_matrix(kappa, n, states):
    zed = 2
    r_0_ang = 0.52917 ## Angstrom
    hy_lambda = zed /(kappa * r_0_ang)
    hbar = sc.constants.hbar
    electron_mass = sc.constants.m_e

    mat_N = [[0]*n for _ in range(n)]
    mat_W = [[0]*n for _ in range(n)]
    mat_C = [[0]*n for _ in range(n)]
    mat_T = [[0]*n for _ in range(n)]

    for i in range(n): #this will give the number of rows
        for j in range(n):  # this will give the number of cols

            j_1 = states[i][0]  
            j_2 = states[j][0]
            k_1 = states[i][1] 
            k_2 = states[j][1]
            m_1 = states[i][2] 
            m_2 = states[j][2]

            jay = states[i][0] + states[j][0]
            kay = states[i][1] + states[j][1]
            em = states[i][2] + states[j][2]


            def N(J, K, M):
                if M == -2:
                    entry_value_N = 0
                else:   
                    factorial_term = factorial((J + K + M +5), exact = True) #if this needs to be faster, set exact to be false
                    wavelength_term = (1/(2*hy_lambda))**((J+K+M+6))
                    coeff = (2*(np.pi)**2) / (M+2)
                    fraction_sum = (1/(K+1) - 1/(K+3) - 1/(K+M+3) + 1/(K+M+5))
            
                    if K%2 ==0:
                        entry_value_N = factorial_term * coeff * wavelength_term * fraction_sum
                    if K%2 ==1:
                        entry_value_N = 0
                return entry_value_N
            
            def W(J, K, M):
                
                value = (electron_charge**(2))*(N(jay, kay, em-1))
                
                return value
            

            def C(J, K, M):
                if M == -2:
                    entry_value_C = 0
                else:   
                    factorial_term = factorial((J + K + M +4), exact = True) #if this needs to be faster, set exact to be false
                    wavelength_term = (1/(2*hy_lambda))**((J+K+M+5))
                    coeff = (8*(np.pi)**2) / (M+2)
                    fraction_sum = (1/(K+1) - 1/(K+M+3))
                    extra_pre = -zed * electron_charge**(2)
            
                    if K%2 ==0:
                        entry_value_C = extra_pre * factorial_term * coeff * wavelength_term * fraction_sum
                    if K%2 ==1:
                        entry_value_C = 0
                return entry_value_C
            
            
            def T(J, K, M, j1, j2, k1, k2, m1, m2):


                coeff = (hbar**2)/(2*electron_mass)
                square_1 = (
                    (hy_lambda**2)*N(jay, kay, em) 
                    - J*hy_lambda*N(jay-1, kay, em) 
                    + j1*j2*N(jay-2, kay, em) 
                    + k1*k2*N(jay,kay-2, em)
                    + m1*m2*N(jay, kay, em-2) 
                )
                square_2 = (
                    -M*hy_lambda*(C(jay, kay, em) - C(jay, kay+2, em-2))
                    +(m1*j2 + m2*j1)*(C(jay-1, kay, em) - C(jay-1, kay+2, em-2))
                    +(m1*k2 + m2*k1)*(C(jay+1, kay, em-2)-C(jay-1,kay, em))
                )
                value = coeff* (
                    2*square_1 + 0.5*square_2
                )
                return value
            
                

            mat_N[i][j] = N(jay, kay, em)
            mat_W[i][j] = W(jay, kay, em)
            mat_C[i][j] = C(jay, kay, em)
            mat_T[i][j] = T(jay, kay, em, j_1, j_2, k_1, k_2, m_1, m_2)
    

    matrices = [mat_N, mat_W, mat_C, mat_T]
    return matrices


In [85]:
N_mat, W_mat, C_mat, T_mat= generate_matrix(test_kappa, chosen_num, chosen_states)

##### Checking Hyllaas' states

In [86]:
## this is to test with the hylleraas states
def correct_N(kappa):
    zed = 2
    r_0_ang = 0.52917 ##Angstrom
    lamb = zed /(kappa * r_0_ang)

    pre = (np.pi )**2 *(lamb)**(-6)

    correct_mat = [
        [pre                     , pre*(35/16)*(lamb)**(-1) , pre*(3/2)*(lamb)**(-2) ],
        [pre*(35/16)*(lamb)**(-1), pre*(6)*(lamb)**(-2)     , pre*(77/16)*(lamb)**(-3)],
        [pre*(3/2)*(lamb)**(-2)  , pre*(77/16)*(lamb)**(-3) ,pre*(9)*(lamb)**(-4)]
    ]
    
    return correct_mat

def correct_W(kappa):
    zed = 2
    r_0_ang = 0.52917 ##Angstrom
    lamb = zed /(kappa * r_0_ang)

    pre = electron_charge**(2)*5*(np.pi**2)/(8*(lamb**5))
    correct_mat = [
        [pre                     , pre*1.6*(lamb)**(-1) , pre*0.9*(lamb)**(-2) ],
        [pre*(1.6)*(lamb)**(-1), pre*(3.5)*(lamb)**(-2)     , pre*(2.4)*(lamb)**(-3)],
        [pre*(0.9)*(lamb)**(-2)  , pre*2.4*(lamb)**(-3) ,pre*(3.9)*(lamb)**(-4)]
    ]
    return correct_mat

def correct_C(kappa):
    zed = 2
    r_0_ang = 0.52917 ##Angstrom
    lamb = zed /(kappa * r_0_ang)

    pre = (-zed) * electron_charge**(2) * 2 * (np.pi**2) *(lamb**(-5))
    correct_mat = [
        [pre                     , pre*(30/16)*(lamb)**(-1) , pre*(3/2)*(lamb)**(-2) ],
        [pre*(30/16)*(lamb)**(-1), pre*(9/2)*(lamb)**(-2)     , pre*(70/16)*(lamb)**(-3)],
        [pre*(3/2)*(lamb)**(-2)  , pre*(70/16)*(lamb)**(-3) ,pre*(9)*(lamb)**(-4)]
    ]
    return correct_mat

def correct_T(kappa):
    zed = 2
    r_0_ang = 0.52917 ##Angstrom
    lamb = zed /(kappa * r_0_ang)
    hbar = sc.constants.hbar

    pre = (hbar**2)*2*(np.pi**2)/(2*sc.constants.m_e*lamb**4)
    correct_mat = [
        [pre                     , pre*(25/16)*(lamb)**(-1) , pre*(3/2)*(lamb)**(-2) ],
        [pre*(25/16)*(lamb)**(-1), pre*(4)*(lamb)**(-2)     , pre*(73/16)*(lamb)**(-3)],
        [pre*(3/2)*(lamb)**(-2)  , pre*(73/16)*(lamb)**(-3) ,pre*(15)*(lamb)**(-4)]
    ]
    return correct_mat

correct_hyll_T = correct_T(test_kappa)
correct_hyll_C = correct_C(test_kappa)
correct_hyll_N = correct_N(test_kappa)
correct_hyll_W = correct_W(test_kappa)

if chosen_states is hyll_states:
    for i in range(chosen_num):
        for j in range(chosen_num):
            assert "%.5f" % N_mat[i][j] == "%.5f" % correct_hyll_N[i][j]

            assert "%.5f" % W_mat[i][j] == "%.5f" % correct_hyll_W[i][j]

            assert "%.5f" % C_mat[i][j] == "%.5f" % correct_hyll_C[i][j]

            assert "%.5f" % T_mat[i][j] == "%.5f" % correct_hyll_T[i][j]

##### Generating $\beta$ 

In [87]:
def find_eig(matrix):
    values, vectors = sc.linalg.eig(matrix)
    values.sort()
    return [values, vectors]

print(find_eig(N_mat)[0])
print()
print(find_eig(N_mat)[1])


[0.01011064+0.j]

[[1.]]


#### Construct $\beta$ from these eigenvalues

In [88]:
##this will give a list with the diagonal matrix and the matrix of eigenvectors
def generate_eig_mats(matrix, n):
    eig_vals, eig_vects = find_eig(matrix)
    diag = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i==j:
                diag[i][j]= float(eig_vals[i])
    
    return [diag, eig_vects]


beta, Y_mat = generate_eig_mats(N_mat, chosen_num)

Y_mat_tran = Y_mat.transpose()



####  Construct $\beta^{\frac{1}{2}}$

In [89]:
def raise_mat_to_power(matrix, power):
    
    length = len(matrix)
    assert length == len(matrix[0]) #checks square matrix

    new_mat = [[0]* length for _ in range(length)]

    for i in range(length):
        for j in range(length):
            
            new_mat[i][j] = (matrix[i][j])**power
    
    return new_mat


In [90]:
test_mat = [
    [1, 2, 3], 
    [2, 3, 4], 
    [1, 4, 3]
]

print(raise_mat_to_power(test_mat, 2))

[[1, 4, 9], [4, 9, 16], [1, 16, 9]]


In [91]:
beta_half_p = raise_mat_to_power(beta, 0.5)
beta_half_m = raise_mat_to_power(beta, -0.5)


#### Generating Hamiltonian

In [92]:
def create_H(T, C, W, n):
    H = [[0]* n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            H[i][j] = T[i][j] + C[i][j] + W[i][j]
    return H

H_mat = np.asarray(create_H(T_mat, C_mat, W_mat, len(T_mat)))
print(H_mat)

[[-22.28593366]]


#### Generating P matrix

In [93]:
N_mat = np.asarray(N_mat)
beta = np.asarray(beta)
beta_half_p = np.asarray(beta_half_p)
beta_half_m = np.array(beta_half_m)
H_mat = np.asarray(H_mat)

print(f'N_mat is a {type(N_mat)}')
print(f'Y is a {type(Y_mat)}')
print(f'Y_T is a {type(Y_mat_tran)}')
print(f'beta is a {type(beta)}')
print(f'beta_1/2 is a {type(beta_half_p)}')
print(f'beta_-1/2 is a {type(beta_half_m)}')
print(f'Hamiltonian is a {type(H_mat)}')

matrices = [N_mat, Y_mat, Y_mat_tran, beta, beta_half_m, beta_half_p, H_mat]

for mat in matrices:
    print(mat.shape)



N_mat is a <class 'numpy.ndarray'>
Y is a <class 'numpy.ndarray'>
Y_T is a <class 'numpy.ndarray'>
beta is a <class 'numpy.ndarray'>
beta_1/2 is a <class 'numpy.ndarray'>
beta_-1/2 is a <class 'numpy.ndarray'>
Hamiltonian is a <class 'numpy.ndarray'>
(1, 1)
(1, 1)
(1, 1)
(1, 1)
(1, 1)
(1, 1)
(1, 1)


In [94]:

def find_energy(B_m, H, Y, YT):
    P = B_m @ YT @ H @ Y @ B_m 

    energies = sc.linalg.eigvals(P)

    energy = min(energies)
    return energies
    

print(find_energy(beta_half_m, H_mat, Y_mat, Y_mat_tran))


[-2204.20658768+0.j]


In [95]:
def correct_energy(kappa):
    r0_ang = 0.52917
    electron_charge = 14.4
    outer = electron_charge**(2) / r0_ang
    inner = (4/(kappa**2)) - (27/(4*kappa))

    return outer * inner

print(correct_energy(test_kappa))

-1115.709507341686
